In [5]:
!pip install openai langchain langchain-community

Defaulting to user installation because normal site-packages is not writeable


In [6]:
from openai import OpenAI
from langchain_community.chat_message_histories import ChatMessageHistory

# Initialize client (LM Studio / local API)
client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="not-needed"
)

history = ChatMessageHistory()
MAX_HISTORY = 10  # limit memory

In [ ]:
def chat():
    while True:
        query = input("\nUser (Enter to skip | type 'exit' to quit): ")

        if query.lower() == "exit":
            print("\nExiting...")
            break

        if not query.strip():
            continue

        print("Input:", query)

        # Save user message
        history.add_user_message(query)

        # Trim history
        if len(history.messages) > MAX_HISTORY * 2:
            history.messages = history.messages[-MAX_HISTORY * 2:]

        # Convert to OpenAI format
        messages_with_history = []
        for m in history.messages:
            role = "user" if m.type == "human" else "assistant"
            messages_with_history.append({
                "role": role,
                "content": m.content
            })

        try:
            stream = client.chat.completions.create(
                model="qwen/qwen3.5-9b",
                messages=messages_with_history,
                stream=True
            )

            full_ai_response = ""

            print("AI:", end=" ", flush=True)

            for chunk in stream:
                delta = chunk.choices[0].delta
                if delta and delta.content:
                    content = delta.content
                    print(content, end="", flush=True)
                    full_ai_response += content

            # Save AI response
            history.add_ai_message(full_ai_response)
            print("\n")

        except Exception as e:
            print(f"\n[ERROR]: {e}")

In [8]:
chat()

Input: hi
AI: 

Hi there! 👋 How can I help you today?

Input: my name is Aj Latina
AI: 

It's nice to meet you, Aj Latina! 👋 How can I help you out today?

Input: no
AI: 

No worries at all! 😊 It's perfectly fine if you don't need anything specific right now. Just let me know if you'd like to chat or ask something later! Have a great day. 🌟

Input: do you know my name?
AI: 

Yes, I do! 😊 You introduced yourself as **Aj Latina** earlier in our conversation. Is there anything else you'd like to talk about?


Exiting...
